# 🧠 Platform Scoring Pipeline — Multi-LLM Candidate Evaluation
**Architecture**: Three specialist LLMs analyse distinct dimension groups, then a Master Aggregator compiles the final report scored out of 1000.

| LLM | Dimensions |
|-----|------------|
| **LLM-1** | Skill Credibility |
| **LLM-2** | Professional Identity + Real World Delivery |
| **LLM-3** | Behavioral Consistency + Professional Track Record |

## 1 · Setup & Imports

In [18]:
import pandas as pd
import json
import os
from openai import OpenAI

endpoint = "https://linked.openai.azure.com/openai/v1/"
deployment_name = "gpt-5.4-mini"
MODEL = "gpt-5.4-mini"
api_key = "CWcXBMalpGqTsxcKFF3movpCCR0xGwpQtMFEfIZEDTEd6oTsFtdlJQQJ99CDACYeBjFXJ3w3AAABACOGYHIY"


client = OpenAI(
    api_key = api_key,
    base_url = endpoint

)

# 2. File paths (adjust if needed)
files = {
    "profile": "candidate_profile (4).csv",
    "resume": "resume_contextual.csv",
    "github_summary": "github_contextual.csv",
    "github_repos": "github_repos (2).csv",
    "hackerrank": "hackerrank_contextual.csv",
    "stackoverflow": "stackoverflow_contextual.csv",
    "linkedin_skills": "linkedin_skills.csv",
    "linkedin_experience": "linkedin_experience.csv",
    "linkedin_education": "linkedin_education.csv"
}

# 3. Read and format the data
def load_and_format_data(file_dict):
    context = ""
    for name, path in file_dict.items():
        try:
            df = pd.read_csv(path)
            # Convert the dataframe to a clean Markdown table string to save tokens
            context += f"### {name.upper()} DATA ###\n"
            context += df.to_markdown(index=False) + "\n\n"
        except Exception as e:
            context += f"### {name.upper()} DATA ###\n[Error loading data: {e}]\n\n"
    return context

candidate_context = load_and_format_data(files)

# 4. Define the prompt for the LLM
system_prompt = """
You are an elite Senior Technical Recruiter and AI Data Analyst.
You are provided with raw data encompassing a candidate's Resume, LinkedIn (Experience, Education, Skills), GitHub, HackerRank, and StackOverflow profiles.

Your goal is to analyze this data holistically and generate a comprehensive candidate evaluation report.

You MUST return your response in two parts:
1. A JSON object containing the detailed analysis.
2. A single CSV row summarizing the candidate, enclosed in a code block.

### JSON Schema Requirements ###
{
  "candidate_name": "String",
  "contact_info": "String",
  "summary": "A brief-sentence executive summary of their profile",
  "years_of_experience": "Float (calculated from LinkedIn/Resume)",
  "top_5_skills": ["List of strings"],
  "platform_insights": {
    "github": "Summary of open-source contributions, languages, and repo quality",
    "hackerrank": "Summary of competitive programming skills and problem-solving abilities",
    "stackoverflow": "Summary of community engagement and reputation"
  },
  "strengths": ["List of top key strengths"],
  "areas_for_improvement": ["List of potential red flags or areas lacking"],
  "recommended_roles": ["List of 2-3 job titles they are best suited for"],
  "overall_score": "Integer out of 1000 based on holistic profile strength",
  "hire_recommendation": "Strong Yes / Yes / Neutral / No"
}

### CSV Schema Requirements ###
Provide a single CSV line with these exact headers:
Name, Years_of_Exp, Top_Skills, GitHub_Summary, HackerRank_Score, Overall_Fitment_Score_1000, Recommendation

Output the CSV inside a ```csv code block.
"""

# 5. Call the OpenAI API
print("Sending data to OpenAI for analysis...")
response = client.chat.completions.create(
    model="gpt-5.4-mini", # Using gpt-4o-mini for fast, cost-effective analysis
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Here is the candidate data:\n\n{candidate_context}"}
    ],
    temperature=0.2, # Low temperature for analytical consistency
)

# 6. Extract and save the results
output_text = response.choices[0].message.content

print("\n--- LLM ANALYSIS COMPLETE ---\n")
print(output_text)



Sending data to OpenAI for analysis...

--- LLM ANALYSIS COMPLETE ---

{
  "candidate_name": "Dewashish Dwivedi",
  "contact_info": "ddwivedi2003@gmail.com | +91 8989862986 | Kanpur, Uttar Pradesh, India",
  "summary": "AI/ML-focused student and Microsoft Student Ambassador with hands-on experience building and deploying Python-based machine learning and computer vision applications, plus strong community leadership and cloud exposure.",
  "years_of_experience": 3.25,
  "top_5_skills": [
    "Python",
    "Machine Learning",
    "Streamlit",
    "Scikit-Learn",
    "Microsoft Azure"
  ],
  "platform_insights": {
    "github": "Active GitHub presence with 15+ repositories, mostly Python-focused ML/data projects and web apps. Repository themes align well with resume claims (healthcare AI, object detection, anomaly detection, data analysis). Codebase quality signals are mixed: several repos are small or config-only, but there are also larger project repos suggesting real implementation wo

In [19]:
# Install dependencies if needed
# !pip install openai pandas tabulate rich -q

import os, json, ast, textwrap, warnings
import pandas as pd
from openai import OpenAI
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')


## 2 · Load All Data Sources

In [20]:
# ── Paths (update if running from a different working directory) ────────────────
BASE = 'data/'   # folder where you placed the CSVs

# Load platform score tables
gh_scores   = pd.read_csv('github_scores.csv').iloc[0].to_dict()
hr_scores   = pd.read_csv('hackerrank_scores.csv').iloc[0].to_dict()
lc_scores   = pd.read_csv('leetcode_scores.csv').iloc[0].to_dict()
li_scores   = pd.read_csv('linkedin_scores.csv').iloc[0].to_dict()
so_scores   = pd.read_csv('stackoverflow_scores.csv').iloc[0].to_dict()

# Load rich profile data
candidate   = pd.read_csv('candidate_profile.csv').iloc[0].to_dict()
li_full     = pd.read_csv('linkedin_profile_full.csv').iloc[0].to_dict()
resume_df   = pd.read_csv('resume_contextual.csv').iloc[0].to_dict()

# Load pre-computed analysis JSON
with open('candidate_analysis.json') as f:
    analysis = json.load(f)

print('✅  All data sources loaded')
print(f'   Candidate: {analysis["candidate_name"]}')

✅  All data sources loaded
   Candidate: Dewashish Dwivedi


## 3 · Helper Utilities

In [21]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict:
    """
    Call the OpenAI API and parse the JSON response.
    Always instructs the model to return strict JSON.
    """
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt}
        ]
    )
    raw = response.choices[0].message.content
    return json.loads(raw)


def render_score_card(title: str, scores: dict, color: str = '#4361ee'):
    """Render a metric score card as styled HTML."""
    rows = ''
    for metric, data in scores.items():
        score = data.get('score', 0)
        max_s = data.get('max', 100)
        pct   = min(100, int(score / max_s * 100))
        bar_color = '#2dc653' if pct >= 70 else ('#f4a261' if pct >= 40 else '#e63946')
        rows += f"""
        <tr>
          <td style='padding:6px 12px;font-size:14px;width:260px'><b>{metric}</b></td>
          <td style='padding:6px 12px;font-size:14px;width:80px;text-align:center'>
            <b>{score}</b>/{max_s}</td>
          <td style='padding:6px 12px;width:200px'>
            <div style='background:#e9ecef;border-radius:4px;height:12px'>
              <div style='background:{bar_color};width:{pct}%;height:12px;border-radius:4px'></div>
            </div>
          </td>
          <td style='padding:6px 12px;font-size:12px;color:#555;max-width:320px'>
            {data.get('insight', '')}</td>
        </tr>"""

    html = f"""
    <div style='font-family:Arial,sans-serif;margin:16px 0'>
      <div style='background:{color};color:white;padding:10px 16px;
                  border-radius:8px 8px 0 0;font-size:16px;font-weight:bold'>
        {title}
      </div>
      <table style='border-collapse:collapse;width:100%;background:#fff;
                    border:1px solid #dee2e6;border-top:none;
                    border-radius:0 0 8px 8px'>
        <thead>
          <tr style='background:#f8f9fa'>
            <th style='padding:8px 12px;text-align:left;font-size:13px'>Metric</th>
            <th style='padding:8px 12px;text-align:center;font-size:13px'>Score</th>
            <th style='padding:8px 12px;font-size:13px'>Progress</th>
            <th style='padding:8px 12px;font-size:13px'>Insight</th>
          </tr>
        </thead>
        <tbody>{rows}</tbody>
      </table>
    </div>"""
    display(HTML(html))


def safe_float(val, default=0.0):
    try:
        v = float(val)
        return 0.0 if (v != v) else v   # NaN check
    except (TypeError, ValueError):
        return default

print('✅  Helpers ready')

✅  Helpers ready


## 4 · Build Contextual Data Packages

In [22]:
# ── Package A: Skill evidence (GitHub + LeetCode + HackerRank + Resume) ─────────
pkg_skill = {
    'candidate_name': analysis['candidate_name'],
    'resume_summary': analysis['summary'],
    'top_skills': analysis['top_5_skills'],
    'resume_text_excerpt': resume_df['resume_text'][:3000],
    'github': {
        'public_repos': safe_float(candidate.get('gh_public_repos')),
        'original_repos': safe_float(candidate.get('gh_original_repos')),
        'total_stars': safe_float(candidate.get('gh_total_stars')),
        'languages': candidate.get('gh_languages', ''),
        'followers': safe_float(candidate.get('gh_followers')),
        'account_age_years': safe_float(gh_scores.get('account_age_years')),
        'commit_frequency_score': safe_float(gh_scores.get('commit_frequency_score')),
        'documentation_quality': safe_float(gh_scores.get('documentation_quality')),
        'language_diversity': safe_float(gh_scores.get('language_diversity')),
        'ci_cd_usage': safe_float(gh_scores.get('ci_cd_usage')),
    },
    'leetcode': {
        'total_solved': safe_float(candidate.get('lc_total_solved')),
        'easy': safe_float(candidate.get('lc_easy_solved')),
        'medium': safe_float(candidate.get('lc_medium_solved')),
        'hard': safe_float(candidate.get('lc_hard_solved')),
        'global_ranking': safe_float(candidate.get('lc_ranking')),
        'contest_rating': safe_float(candidate.get('lc_contest_rating')),
        'star_rating': safe_float(candidate.get('lc_star_rating')),
        'languages': candidate.get('lc_languages', ''),
        'top_topics': candidate.get('lc_top_topics', ''),
        'problems_solved_score': safe_float(lc_scores.get('problems_solved_score')),
        'global_ranking_score': safe_float(lc_scores.get('global_ranking_score')),
        'difficulty_distribution': safe_float(lc_scores.get('difficulty_distribution')),
    },
    'hackerrank': {
        'total_badges': safe_float(candidate.get('hr_total_badges')),
        'problem_solving_stars': safe_float(candidate.get('hr_problem_solving_stars')),
        'python_stars': safe_float(candidate.get('hr_python_stars')),
        'cpp_stars': safe_float(candidate.get('hr_c++_stars')),
        'java_stars': safe_float(candidate.get('hr_java_stars')),
        'skills_raw': candidate.get('hr_skills_raw', ''),
        'skill_certificates_score': safe_float(hr_scores.get('skill_certificates_score')),
        'avg_stars_score': safe_float(hr_scores.get('avg_stars_score')),
        'domain_score_quality': safe_float(hr_scores.get('domain_score_quality')),
    },
    'certifications': str(li_full.get('certification', '')),
    'platform_insights': analysis['platform_insights'],
}

# ── Package B: Professional identity (LinkedIn + StackOverflow + GitHub) ─────────
pkg_identity = {
    'candidate_name': analysis['candidate_name'],
    'linkedin': {
        'connections': safe_float(li_full.get('connections')),
        'followers': safe_float(li_full.get('followers')),
        'profile_headline': li_full.get('profile_headline', ''),
        'description': str(li_full.get('description', ''))[:1500],
        'skills': str(li_full.get('skills', ''))[:800],
        'education': str(li_full.get('education', ''))[:600],
        'certifications': str(li_full.get('certification', ''))[:600],
        'profile_completeness': safe_float(li_scores.get('profile_completeness')),
        'skill_endorsement_credibility': safe_float(li_scores.get('skill_endorsement_credibility')),
        'headline_keyword_density': safe_float(li_scores.get('headline_keyword_density')),
        'education_verification_score': safe_float(li_scores.get('education_verification_score')),
        'network_size_quality': safe_float(li_scores.get('network_size_quality')),
        'company_prestige_score': safe_float(li_scores.get('company_prestige_score')),
        'recommendations_given': str(li_full.get('recommendations_given', '')),
        'recommendations_received': str(li_full.get('recommendations_received', '')),
    },
    'stackoverflow': {
        'reputation': safe_float(candidate.get('so_reputation')),
        'answer_count': safe_float(candidate.get('so_answer_count')),
        'question_count': safe_float(candidate.get('so_question_count')),
        'accepted_answers': safe_float(candidate.get('so_accepted_answers')),
        'top_tags': str(candidate.get('so_top_tags', '')),
        'account_age_score': safe_float(so_scores.get('account_age_score')),
        'reputation_score': safe_float(so_scores.get('reputation_score')),
    },
    'github': {
        'profile_readme': safe_float(gh_scores.get('profile_readme')),
        'collaboration_network': safe_float(gh_scores.get('collaboration_network')),
        'account_age_years': safe_float(gh_scores.get('account_age_years')),
    },
    'experience': str(li_full.get('experience', ''))[:2000],
    'freelance_work': [
        'Music Producer at SoundBetter (Self-employed, Jan 2021 – Feb 2023) — produced 50+ tracks',
        'Music Producer at BandLab (Self-employed, Oct 2020 – Feb 2023)',
        'Course Builder at Robokart.com (Freelance, Mar 2019 – Jun 2022)',
    ],
    'community_activities': [
        'Microsoft Student Ambassador — hosted 5+ technical webinars on ML, AI, SQL, GIT, .NET',
        'ICPC Asia Kanpur 2024 Participant',
        '.NET Conf Student Zone Member 2025',
        '.NET Foundation Member',
        'Firebase Communities Group Member',
    ],
}

# ── Package C: Behavioral signals + career track (LinkedIn time-series) ──────────
pkg_behavior = {
    'candidate_name': analysis['candidate_name'],
    'linkedin_activity': {
        'activity_frequency_score': safe_float(li_scores.get('activity_frequency_score')),
        'profile_age_years': safe_float(li_scores.get('profile_age_years')),
        'profile_age_score': safe_float(li_scores.get('profile_age_score')),
        'volunteer_certification_activity': safe_float(li_scores.get('volunteer_certification_activity')),
        'content_quality_score': safe_float(li_scores.get('content_quality_score')),
        'open_to_work_signal': safe_float(li_scores.get('open_to_work_signal')),
    },
    'github_activity': {
        'commit_frequency_score': safe_float(gh_scores.get('commit_frequency_score')),
        'contribution_graph_density_score': safe_float(gh_scores.get('contribution_graph_density_score')),
        'project_longevity': safe_float(gh_scores.get('project_longevity')),
        'account_created': candidate.get('gh_account_created', ''),
    },
    'leetcode_activity': {
        'total_solved': safe_float(lc_scores.get('lc_total_solved')),
        'streak_quality': safe_float(lc_scores.get('streak_quality')),
        'contest_participation_score': safe_float(lc_scores.get('contest_participation_score')),
        'category_coverage': safe_float(lc_scores.get('category_coverage')),
    },
    'career_history': [
        {'title': 'Threat Advisor',            'company': 'Microsoft', 'type': 'Apprenticeship', 'start': '3-2026',  'end': 'Present', 'size': '10001+'},
        {'title': 'Beta Microsoft Student Ambassador', 'company': 'Microsoft', 'type': 'Apprenticeship', 'start': '6-2024',  'end': 'Present', 'size': '10001+'},
        {'title': 'Azure Experience Insiders (AXI)', 'company': 'Microsoft', 'type': 'Apprenticeship', 'start': '2-2025',  'end': '3-2026',  'size': '10001+'},
        {'title': 'Alpha Microsoft Student Ambassador','company': 'Microsoft','type': 'Apprenticeship', 'start': '12-2023', 'end': '6-2024',  'size': '10001+'},
        {'title': 'Microsoft Student Ambassador', 'company': 'Microsoft',       'type': 'Apprenticeship', 'start': '11-2023', 'end': '12-2023', 'size': '10001+'},
        {'title': 'Member',                      'company': '.NET Foundation',  'type': '',               'start': '2-2026',  'end': 'Present', 'size': '2-10'},
        {'title': 'Music Producer',              'company': 'SoundBetter',      'type': 'Self-employed',  'start': '1-2021',  'end': '2-2023',  'size': '2-10'},
        {'title': 'Course Builder',              'company': 'Robokart.com',     'type': 'Freelance',      'start': '3-2019',  'end': '6-2022',  'size': '11-50'},
        {'title': 'Teacher',                     'company': 'Google Skillshop', 'type': 'Part-time',      'start': '3-2018',  'end': '11-2021', 'size': '501-1000'},
    ],
    'education': {
        'degree': 'B.Tech AI & ML',
        'institution': 'PSIT Kanpur',
        'started': '2022',
        'ends': '2026',
        'current_grade': '75.1%',
    },
    'professional_track_scores': {
        'career_progression_score_raw': safe_float(li_scores.get('career_progression_trajectory')),
        'employment_stability_score_raw': safe_float(li_scores.get('employment_consistency_score')),
        'company_quality_score_raw': safe_float(li_scores.get('company_prestige_score')),
        'avg_tenure_months': safe_float(li_scores.get('avg_tenure_months')),
        'job_switching_frequency': safe_float(li_scores.get('job_switching_frequency')),
        'job_switching_score': safe_float(li_scores.get('job_switching_score')),
    },
    'certifications_list': [
        'Data Analysis with Python – freeCodeCamp (2025)',
        'ICPC 2024 Participation – ICPC (2024)',
        'Programming with Python – OpenEDG Python Institute (2025)',
        'AI Foundations: Machine Learning – LinkedIn Learning (2025)',
        'Career Essentials in Software Development – Microsoft & LinkedIn (2024)',
        'Building an Adaptability Mindset in the Age of AI – LinkedIn (2026)',
        'Investing in Human Skills in the Age of AI – LinkedIn (2026)',
    ],
}

print('✅  Data packages A, B, C built')

✅  Data packages A, B, C built


## 5 · LLM-1 — Skill Credibility
**Metrics**: Technical Depth Score · Domain Expertise Score · Competitive Performance Score · Certification Credibility Score

In [23]:
SYSTEM_LLM1 = textwrap.dedent("""
You are an expert technical recruiter and skill evaluator specialising in AI/ML and software engineering.
You receive structured candidate data from GitHub, LeetCode, HackerRank and resume.

Evaluate FOUR skill credibility metrics each scored out of 250 (total dimension max = 1000 if combined,
but for this dimension each metric is out of 250).

Metric definitions:
1. Technical Depth Score (0-250): Depth of technical knowledge evidenced by project complexity,
   algorithm difficulty, code quality, use of advanced frameworks (MLOps, deep learning, pipelines).
2. Domain Expertise Score (0-250): Breadth and depth in the candidate's primary domain (AI/ML).
   Consider specialisation, diversity of tools, production-grade evidence.
3. Competitive Performance Score (0-250): Performance on competitive platforms — LeetCode rankings,
   HackerRank stars, contest participation, difficulty of solved problems.
4. Certification Credibility Score (0-250): Relevance, recency, issuer prestige, and number of
   certifications relative to the candidate's career stage.

Return ONLY valid JSON matching this exact schema (no markdown fences):
{
  "dimension": "Skill Credibility",
  "dimension_total": <int 0-1000>,
  "metrics": {
    "Technical Depth Score":         {"score": <int 0-250>, "max": 250, "insight": "<2-sentence rationale>"},
    "Domain Expertise Score":        {"score": <int 0-250>, "max": 250, "insight": "<2-sentence rationale>"},
    "Competitive Performance Score": {"score": <int 0-250>, "max": 250, "insight": "<2-sentence rationale>"},
    "Certification Credibility Score":{"score": <int 0-250>, "max": 250, "insight": "<2-sentence rationale>"}
  },
  "strengths": ["<str>", "<str>"],
  "gaps": ["<str>", "<str>"],
  "overall_comment": "<3 sentence summary>"
}
""")

USER_LLM1 = f"""
Candidate data for Skill Credibility evaluation:

{json.dumps(pkg_skill, indent=2, default=str)}

Score each metric rigorously. Penalise heavily for: zero stars/forks on GitHub, no contest rating,
no HackerRank rank, basic-only LeetCode problems. Reward for: production deployments, advanced
algorithms, high-prestige certifications, strong domain specialisation.
"""

print('🔄  Calling LLM-1 (Skill Credibility)...')
result_llm1 = call_llm(SYSTEM_LLM1, USER_LLM1)
print(f'✅  LLM-1 done | Dimension Total: {result_llm1["dimension_total"]} / 1000')

🔄  Calling LLM-1 (Skill Credibility)...
✅  LLM-1 done | Dimension Total: 518 / 1000


In [24]:
render_score_card(
    '📊 LLM-1 · Skill Credibility',
    result_llm1['metrics'],
    color='#4361ee'
)
display(Markdown(f"**Overall Comment:** {result_llm1['overall_comment']}"))
display(Markdown("**Strengths:** " + " · ".join(result_llm1['strengths'])))
display(Markdown("**Gaps:** " + " · ".join(result_llm1['gaps'])))

Metric,Score,Progress,Insight
Technical Depth Score,168/250,,"The candidate shows practical technical depth through end-to-end ML pipelines, Azure/OpenAI integrations, YOLOv8 deployment, and measurable outcomes like OCR accuracy and reduced setup time. However, the GitHub footprint suggests mostly small-scale experimentation with limited engineering maturity, no CI/CD evidence, and low external validation of code quality."
Domain Expertise Score,176/250,,"There is clear alignment with AI/ML through coursework, projects in healthcare AI, churn prediction, object detection, and use of tools like Scikit-Learn, TensorFlow, Streamlit, Azure, and GenAI. The breadth is good for an aspiring ML engineer, but the evidence still leans toward student-level implementation rather than sustained production-grade specialization."
Competitive Performance Score,132/250,,"LeetCode performance is respectable with 130 solved problems, including some hard problems and exposure to advanced topics like DP, Union-Find, and monotonic stacks. That said, the lack of contest rating, no HackerRank rank, and only moderate platform signals keep this below strong competitive-programming credibility."
Certification Credibility Score,42/250,,"The certifications listed are mostly low-prestige, short-form learning credentials from LinkedIn Learning, FreeCodeCamp, OpenEDG, and Microsoft-branded career essentials rather than rigorous industry certifications. They are also recent but not especially selective, so they add some signal of initiative without materially strengthening credibility."


**Overall Comment:** This candidate looks like a promising early-career ML engineer with real hands-on project experience and decent algorithmic fundamentals. The strongest evidence comes from applied AI/ML work and a reasonable LeetCode profile, but public proof of engineering maturity and external validation remains limited. Overall credibility is solid for an aspiring student-level profile, but not yet strong enough to indicate advanced professional depth.

**Strengths:** Strong practical AI/ML project exposure with Azure/OpenAI, Streamlit, YOLOv8, and measurable model outcomes. · Good algorithmic foundation demonstrated by 130 LeetCode solves and multi-language problem-solving practice.

**Gaps:** GitHub evidence is weak for credibility: zero stars/forks, limited documentation quality, and no CI/CD or production engineering signals. · Certifications and competitive-platform signals are modest, with no contest rating, no HackerRank rank, and mostly non-selective certificates.

## 6 · LLM-2 — Professional Identity + Real World Delivery
**Identity metrics**: Profile Completeness · Cross-Platform Consistency · Account Authenticity  
**Delivery metrics**: Freelance Delivery · Community Impact

In [25]:
SYSTEM_LLM2 = textwrap.dedent("""
You are a senior talent branding analyst and professional identity verifier.
You evaluate candidates on two grouped sub-dimensions:

A) PROFESSIONAL IDENTITY (3 metrics, each 0-150, sub-total 0-450):
   1. Profile Completeness Score (0-150): Completeness of LinkedIn, GitHub bios, README, headline,
      about section, skills endorsements, education, and contact details.
   2. Cross-Platform Consistency Score (0-150): Consistency of name, skills, roles, and timelines
      across LinkedIn, GitHub, resume, StackOverflow and HackerRank.
   3. Account Authenticity Score (0-150): Indicators of genuine long-lived accounts — account ages,
      organic follower/connection growth, absence of obvious fabrication.

B) REAL WORLD DELIVERY (2 metrics, each 0-275, sub-total 0-550):
   4. Freelance Delivery Score (0-275): Evidence of paid/freelance work delivered — recency,
      volume, diversity, client feedback signals, and employment type.
   5. Community Impact Score (0-275): Open-source contributions, webinars hosted, events organised,
      talks given, blog posts, teaching, ambassador roles, and reach.

Grand total for this dimension = sum of all 5 metrics (max 1000).

Return ONLY valid JSON (no markdown fences):
{
  "dimension": "Professional Identity + Real World Delivery",
  "dimension_total": <int 0-1000>,
  "sub_dimensions": {
    "Professional Identity": <int 0-450>,
    "Real World Delivery": <int 0-550>
  },
  "metrics": {
    "Profile Completeness Score":        {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Cross-Platform Consistency Score":  {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Account Authenticity Score":        {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Freelance Delivery Score":          {"score": <int 0-275>, "max": 275, "insight": "<2-sentence rationale>"},
    "Community Impact Score":            {"score": <int 0-275>, "max": 275, "insight": "<2-sentence rationale>"}
  },
  "strengths": ["<str>", "<str>"],
  "gaps": ["<str>", "<str>"],
  "overall_comment": "<3 sentence summary>"
}
""")

USER_LLM2 = f"""
Candidate data for Professional Identity + Real World Delivery evaluation:

{json.dumps(pkg_identity, indent=2, default=str)}

Evaluate rigorously. Reward for: genuine ambassador role, high-follower network, consistent
narrative, active community leadership, delivered freelance projects. Penalise for:
missing StackOverflow activity, sparse GitHub bio, no cross-platform skill alignment.
"""

print('🔄  Calling LLM-2 (Professional Identity + Real World Delivery)...')
result_llm2 = call_llm(SYSTEM_LLM2, USER_LLM2)
print(f'✅  LLM-2 done | Dimension Total: {result_llm2["dimension_total"]} / 1000')

🔄  Calling LLM-2 (Professional Identity + Real World Delivery)...
✅  LLM-2 done | Dimension Total: 649 / 1000


In [26]:
render_score_card(
    '📋 LLM-2 · Professional Identity + Real World Delivery',
    result_llm2['metrics'],
    color='#7209b7'
)
display(Markdown(f"**Sub-dimension totals →** Professional Identity: {result_llm2['sub_dimensions']['Professional Identity']}/450 · Real World Delivery: {result_llm2['sub_dimensions']['Real World Delivery']}/550"))
display(Markdown(f"**Overall Comment:** {result_llm2['overall_comment']}"))
display(Markdown("**Strengths:** " + " · ".join(result_llm2['strengths'])))
display(Markdown("**Gaps:** " + " · ".join(result_llm2['gaps'])))

Metric,Score,Progress,Insight
Profile Completeness Score,112/150,,"LinkedIn is fairly complete with a detailed about section, headline, education, skills, and contact/network signals, and the profile shows a clear ambassador-oriented narrative. GitHub is comparatively sparse beyond a README signal, and the absence of richer bios, repositories, or contact details limits the completeness score."
Cross-Platform Consistency Score,98/150,,"The core story is broadly consistent across LinkedIn, GitHub age, and the listed community activities: AI/ML, Microsoft Student Ambassador, and competitive programming. However, there is weak evidence of alignment with StackOverflow, and the freelance history in music production/course building does not strongly match the current technical branding, reducing consistency."
Account Authenticity Score,139/150,,"The account shows strong authenticity indicators: a large but plausible LinkedIn network, a multi-year GitHub account, and a profile that reads like an organically developed student/ambassador identity rather than a fabricated one. The only notable caution is some profile data irregularity and future-dated employment entries, which slightly temper confidence."
Freelance Delivery Score,188/275,,"There is credible evidence of paid delivery through multiple self-employed/freelance roles over several years, including a concrete output claim of 50+ tracks and a course-building engagement. The work is real-world and sustained, but it is concentrated in music and educational content rather than directly in the candidate's current AI/ML positioning, so the score is solid but not top-tier."
Community Impact Score,112/275,,"The candidate shows meaningful community participation through Microsoft Student Ambassador activity, webinar hosting, and membership in technical communities such as .NET Conf Student Zone and Firebase groups. Impact appears active but modest in scale and reach, with limited evidence of public artifacts, open-source contribution depth, or broad audience influence."


**Sub-dimension totals →** Professional Identity: 349/450 · Real World Delivery: 300/550

**Overall Comment:** This candidate presents as a genuine, active student technologist with credible ambassador participation and a reasonably strong public identity. The profile is strongest in LinkedIn presence and community engagement, while technical proof on GitHub and StackOverflow is comparatively thin. Overall, the evidence supports a solid but not exceptional score, with authenticity and delivery stronger than cross-platform technical consistency.

**Strengths:** Strong ambassador and community-facing identity with high LinkedIn network size and repeated Microsoft-affiliated involvement. · Demonstrated real-world delivery history through multiple freelance/self-employed roles and concrete output claims.

**Gaps:** Sparse StackOverflow and limited GitHub depth reduce proof of technical community contribution. · Cross-platform branding is not fully aligned, especially where freelance history and technical positioning diverge.

## 7 · LLM-3 — Behavioral Consistency + Professional Track Record
**Behavioral metrics**: Activity Consistency · Learning Velocity · Activity Recency  
**Track record metrics** (weighted): Career Progression 25% · Employment Stability 50% · Company Quality 25%

In [27]:
SYSTEM_LLM3 = textwrap.dedent("""
You are an experienced career analyst and behavioural recruiter.
You evaluate two sub-dimensions:

A) BEHAVIORAL CONSISTENCY (3 metrics, each 0-150, sub-total 0-450):
   1. Activity Consistency Score (0-150): Regularity and sustained frequency of coding, posting,
      and professional activity across platforms over time.
   2. Learning Velocity Score (0-150): Speed of skill acquisition evidenced by certifications earned,
      problem-solving progression, new technologies adopted per year.
   3. Activity Recency Score (0-150): How recently the candidate was active — recent commits,
      certifications, and job/role transitions.

B) PROFESSIONAL TRACK RECORD (3 metrics, sub-total 0-550, WEIGHTED internally):
   4. Career Progression Score   (weight 25%, max contribution 137.5 → round to int):
      Direction and quality of role progression — junior→senior, lateral, or downgrade.
   5. Employment Stability Score (weight 50%, max contribution 275):
      Tenure length, voluntary vs forced changes, gaps between jobs.
   6. Company Quality Score      (weight 25%, max contribution 137.5 → round to int):
      Prestige and growth-stage of employers — FAANG/tier-1, startup, freelance, etc.

   For each track record metric, first compute a raw 0-100 quality score, then apply the weight
   and scale to the max contribution to get the final score.
   Report: raw score (0-100), weighted score (0-max contribution), and max in the output.

Grand total for this dimension = Behavioral sub-total + Professional Track Record sub-total (max 1000).

Return ONLY valid JSON (no markdown fences):
{
  "dimension": "Behavioral Consistency + Professional Track Record",
  "dimension_total": <int 0-1000>,
  "sub_dimensions": {
    "Behavioral Consistency": <int 0-450>,
    "Professional Track Record": <int 0-550>
  },
  "metrics": {
    "Activity Consistency Score": {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Learning Velocity Score":    {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Activity Recency Score":     {"score": <int 0-150>, "max": 150, "insight": "<2-sentence rationale>"},
    "Career Progression Score":   {
        "raw_score": <int 0-100>, "weight": "25%",
        "score": <int 0-138>, "max": 138, "insight": "<2-sentence rationale>"
    },
    "Employment Stability Score": {
        "raw_score": <int 0-100>, "weight": "50%",
        "score": <int 0-275>, "max": 275, "insight": "<2-sentence rationale>"
    },
    "Company Quality Score":      {
        "raw_score": <int 0-100>, "weight": "25%",
        "score": <int 0-137>, "max": 137, "insight": "<2-sentence rationale>"
    }
  },
  "strengths": ["<str>", "<str>"],
  "gaps": ["<str>", "<str>"],
  "overall_comment": "<3 sentence summary>"
}
""")

USER_LLM3 = f"""
Candidate data for Behavioral Consistency + Professional Track Record evaluation:

{json.dumps(pkg_behavior, indent=2, default=str)}

Important context: the candidate is a final-year B.Tech student (2022-2026), so all roles
are student/apprenticeship-level. Calibrate expectations accordingly. Penalise for:
role gaps, very short tenures without explanation, low contribution density.
Reward for: consistent upward progression within Microsoft ecosystem, certification
cadence, recent platform activity.
"""

print('🔄  Calling LLM-3 (Behavioral Consistency + Professional Track Record)...')
result_llm3 = call_llm(SYSTEM_LLM3, USER_LLM3)
print(f'✅  LLM-3 done | Dimension Total: {result_llm3["dimension_total"]} / 1000')

🔄  Calling LLM-3 (Behavioral Consistency + Professional Track Record)...
✅  LLM-3 done | Dimension Total: 756 / 1000


In [28]:
render_score_card(
    '🧭 LLM-3 · Behavioral Consistency + Professional Track Record',
    result_llm3['metrics'],
    color='#e63946'
)
display(Markdown(f"**Sub-dimension totals →** Behavioral Consistency: {result_llm3['sub_dimensions']['Behavioral Consistency']}/450 · Professional Track Record: {result_llm3['sub_dimensions']['Professional Track Record']}/550"))
display(Markdown(f"**Overall Comment:** {result_llm3['overall_comment']}"))
display(Markdown("**Strengths:** " + " · ".join(result_llm3['strengths'])))
display(Markdown("**Gaps:** " + " · ".join(result_llm3['gaps'])))

Metric,Score,Progress,Insight
Activity Consistency Score,118/150,,"LinkedIn activity is strong and sustained, with a high activity frequency score and long profile age indicating ongoing professional presence. GitHub and LeetCode signals are more moderate, so consistency is good overall but not equally strong across all platforms."
Learning Velocity Score,101/150,,"The candidate shows solid learning momentum through multiple certifications in 2024-2026, including Python, AI, and software development credentials. Problem-solving breadth is decent, but GitHub contribution density and contest participation are limited, which caps the velocity score."
Activity Recency Score,100/150,,"Recent certifications in 2025 and 2026, plus current Microsoft and .NET Foundation roles, indicate strong recent professional activity. GitHub recency is weaker due to low contribution density, but the overall profile remains active and current."
Career Progression Score,110/138,,"The candidate shows clear upward progression within the Microsoft ecosystem, moving through multiple ambassador/apprenticeship roles into a more advanced Threat Advisor position. For a final-year student, this is a strong trajectory with increasing responsibility and continuity in a prestigious environment."
Employment Stability Score,187/275,,"Average tenure is reasonable for student-level roles, and the Microsoft sequence suggests sustained engagement rather than random switching. However, there are several short tenures and overlapping roles, which introduce some instability and make the timeline less clean."
Company Quality Score,121/137,,"Microsoft materially strengthens the profile, and repeated involvement there is a strong signal of employer quality and selectivity. Additional exposure through Google Skillshop and .NET Foundation adds credibility, though some freelance/self-employed work is naturally lower prestige."


**Sub-dimension totals →** Behavioral Consistency: 319/450 · Professional Track Record: 437/550

**Overall Comment:** This is a strong final-year student profile with unusually good brand quality and a clear progression path inside Microsoft. Behavioral signals are solid, especially on LinkedIn and certifications, but technical activity depth on GitHub and LeetCode is only moderate. Overall, the candidate looks promising for early-career roles, with the main caution being timeline complexity and limited public coding density.

**Strengths:** Strong upward progression and repeated selection within the Microsoft ecosystem. · Good recent learning cadence supported by multiple certifications and current student leadership/apprenticeship activity.

**Gaps:** GitHub contribution density is low, limiting evidence of sustained hands-on coding output. · Several short or overlapping student roles reduce employment stability clarity.

## 8 · Master Aggregation — Final Score out of 1000

In [29]:
# ── Raw dimension scores from each LLM ─────────────────────────────────────────
D1 = result_llm1['dimension_total']   # Skill Credibility          / 1000
D2 = result_llm2['dimension_total']   # Prof. Identity + Delivery  / 1000
D3 = result_llm3['dimension_total']   # Behavioral + Track Record  / 1000

# Equal weight across three LLM dimensions → final score scaled to 1000
FINAL_SCORE = round((D1 + D2 + D3) / 3)

print(f'Dimension 1 — Skill Credibility:                 {D1:>5} / 1000')
print(f'Dimension 2 — Professional Identity + Delivery:  {D2:>5} / 1000')
print(f'Dimension 3 — Behavioral + Track Record:         {D3:>5} / 1000')
print(f'{"─"*52}')
print(f'FINAL PLATFORM SCORE (avg):                      {FINAL_SCORE:>5} / 1000')

Dimension 1 — Skill Credibility:                   518 / 1000
Dimension 2 — Professional Identity + Delivery:    649 / 1000
Dimension 3 — Behavioral + Track Record:           756 / 1000
────────────────────────────────────────────────────
FINAL PLATFORM SCORE (avg):                        641 / 1000


## 9 · Comprehensive Report Generation

In [30]:
SYSTEM_MASTER = textwrap.dedent("""
You are the Chief Talent Analyst producing a definitive candidate evaluation report
for a recruitment platform. Synthesise three specialist LLM assessments into one
coherent, executive-quality report.

The report must include:
- Executive Summary (3-4 sentences)
- Scoring Table (all metrics with scores)
- Dimension-by-Dimension Analysis
- Key Strengths (top 4)
- Critical Gaps (top 3)
- Recommended Roles
- Hire Recommendation with rationale

Return ONLY valid JSON (no markdown fences) matching:
{
  "report_title": "Platform Scoring Report — <name>",
  "candidate": "<name>",
  "final_score": <int>,
  "max_score": 1000,
  "tier": "<one of: Elite / Strong / Promising / Developing / Weak>",
  "executive_summary": "<string>",
  "dimension_breakdown": [
    {"dimension": "Skill Credibility", "score": <int>, "max": 1000, "summary": "<str>"},
    {"dimension": "Professional Identity + Real World Delivery", "score": <int>, "max": 1000, "summary": "<str>"},
    {"dimension": "Behavioral Consistency + Professional Track Record", "score": <int>, "max": 1000, "summary": "<str>"}
  ],
  "key_strengths": ["<str>", "<str>", "<str>", "<str>"],
  "critical_gaps": ["<str>", "<str>", "<str>"],
  "recommended_roles": ["<str>", "<str>", "<str>"],
  "hire_recommendation": "Yes / No / Conditional",
  "hire_rationale": "<2-3 sentence rationale>",
  "development_roadmap": ["<str>", "<str>", "<str>"]
}
""")

USER_MASTER = f"""
Final scores:
  Skill Credibility:                 {D1} / 1000
  Professional Identity + Delivery:  {D2} / 1000
  Behavioral + Track Record:         {D3} / 1000
  FINAL SCORE:                       {FINAL_SCORE} / 1000

LLM-1 full report:
{json.dumps(result_llm1, indent=2)}

LLM-2 full report:
{json.dumps(result_llm2, indent=2)}

LLM-3 full report:
{json.dumps(result_llm3, indent=2)}

Background context:
{json.dumps(analysis, indent=2)}

Produce the final comprehensive report now.
"""

print('🔄  Generating Master Report...')
master_report = call_llm(SYSTEM_MASTER, USER_MASTER)
print(f'✅  Master Report generated | Tier: {master_report["tier"]} | Hire: {master_report["hire_recommendation"]}')

🔄  Generating Master Report...
✅  Master Report generated | Tier: Strong | Hire: Conditional


In [31]:
# ── Render the full report as styled HTML ──────────────────────────────────────
tier_colors = {
    'Elite': '#2dc653', 'Strong': '#4cc9f0', 'Promising': '#4361ee',
    'Developing': '#f4a261', 'Weak': '#e63946'
}
tier = master_report.get('tier', 'Developing')
tc   = tier_colors.get(tier, '#888')
score = master_report['final_score']
pct   = round(score / 10)

dim_rows = ''.join([
    f"""<tr>
      <td style='padding:8px 14px'>{d['dimension']}</td>
      <td style='padding:8px 14px;text-align:center'><b>{d['score']}</b>/1000</td>
      <td style='padding:8px 14px'>
        <div style='background:#e9ecef;border-radius:4px;height:10px'>
          <div style='background:{tc};width:{round(d['score']/10)}%;height:10px;border-radius:4px'></div>
        </div></td>
      <td style='padding:8px 14px;font-size:12px;color:#555'>{d['summary']}</td>
    </tr>"""
    for d in master_report['dimension_breakdown']
])

strengths_li = ''.join([f'<li>{s}</li>' for s in master_report['key_strengths']])
gaps_li      = ''.join([f'<li>{g}</li>' for g in master_report['critical_gaps']])
roles_li     = ''.join([f'<li>{r}</li>' for r in master_report['recommended_roles']])
roadmap_li   = ''.join([f'<li>{r}</li>' for r in master_report['development_roadmap']])

hire_badge = {
    'Yes': ('✅ YES', '#2dc653'),
    'No': ('❌ NO', '#e63946'),
    'Conditional': ('⚠️ CONDITIONAL', '#f4a261')
}.get(master_report['hire_recommendation'], ('—', '#888'))

html_report = f"""
<div style='font-family:Arial,sans-serif;max-width:960px;margin:0 auto;border:1px solid #dee2e6;border-radius:12px;overflow:hidden'>

  <!-- Header -->
  <div style='background:linear-gradient(135deg,#0f0c29,#302b63,#24243e);color:white;padding:24px 28px'>
    <h1 style='margin:0;font-size:22px'>🧠 {master_report['report_title']}</h1>
    <p style='margin:6px 0 0;opacity:0.8;font-size:13px'>Powered by 3-LLM Specialist Pipeline · Model: {MODEL}</p>
  </div>

  <!-- Score banner -->
  <div style='background:#f8f9fa;padding:20px 28px;display:flex;align-items:center;gap:32px;border-bottom:1px solid #dee2e6'>
    <div style='text-align:center'>
      <div style='font-size:56px;font-weight:900;color:{tc};line-height:1'>{score}</div>
      <div style='font-size:13px;color:#888'>out of 1000</div>
    </div>
    <div>
      <div style='font-size:18px;font-weight:bold;color:{tc};margin-bottom:4px'>{tier} Candidate</div>
      <div style='background:#e9ecef;border-radius:6px;height:16px;width:320px'>
        <div style='background:{tc};width:{pct}%;height:16px;border-radius:6px'></div>
      </div>
      <div style='font-size:12px;color:#888;margin-top:4px'>{pct}th percentile equivalent</div>
    </div>
    <div style='margin-left:auto;text-align:center'>
      <div style='font-size:13px;color:#555;margin-bottom:6px'>HIRE RECOMMENDATION</div>
      <div style='background:{hire_badge[1]};color:white;padding:8px 18px;border-radius:20px;font-weight:bold;font-size:14px'>
        {hire_badge[0]}</div>
    </div>
  </div>

  <div style='padding:24px 28px'>

    <!-- Executive Summary -->
    <h2 style='font-size:16px;color:#333;border-bottom:2px solid {tc};padding-bottom:6px'>📌 Executive Summary</h2>
    <p style='color:#444;line-height:1.7'>{master_report['executive_summary']}</p>

    <!-- Dimension Breakdown -->
    <h2 style='font-size:16px;color:#333;border-bottom:2px solid {tc};padding-bottom:6px;margin-top:24px'>📊 Dimension Breakdown</h2>
    <table style='width:100%;border-collapse:collapse;font-size:14px'>
      <thead>
        <tr style='background:#f8f9fa'>
          <th style='padding:8px 14px;text-align:left'>Dimension</th>
          <th style='padding:8px 14px;text-align:center'>Score</th>
          <th style='padding:8px 14px;min-width:160px'>Progress</th>
          <th style='padding:8px 14px;text-align:left'>Summary</th>
        </tr>
      </thead>
      <tbody>{dim_rows}</tbody>
    </table>

    <!-- Two-col layout: Strengths + Gaps -->
    <div style='display:flex;gap:24px;margin-top:24px'>
      <div style='flex:1;background:#f0fff4;border:1px solid #b7eec8;border-radius:8px;padding:16px'>
        <h3 style='margin:0 0 10px;font-size:14px;color:#2d6a4f'>✅ Key Strengths</h3>
        <ul style='margin:0;padding-left:18px;font-size:13px;color:#1b4332;line-height:1.8'>{strengths_li}</ul>
      </div>
      <div style='flex:1;background:#fff5f5;border:1px solid #f5c2c7;border-radius:8px;padding:16px'>
        <h3 style='margin:0 0 10px;font-size:14px;color:#842029'>⚠️ Critical Gaps</h3>
        <ul style='margin:0;padding-left:18px;font-size:13px;color:#58151c;line-height:1.8'>{gaps_li}</ul>
      </div>
    </div>

    <!-- Roles + Roadmap -->
    <div style='display:flex;gap:24px;margin-top:20px'>
      <div style='flex:1;background:#f0f4ff;border:1px solid #c5d0f5;border-radius:8px;padding:16px'>
        <h3 style='margin:0 0 10px;font-size:14px;color:#1a237e'>🎯 Recommended Roles</h3>
        <ul style='margin:0;padding-left:18px;font-size:13px;color:#283593;line-height:1.8'>{roles_li}</ul>
      </div>
      <div style='flex:1;background:#fffde7;border:1px solid #fff59d;border-radius:8px;padding:16px'>
        <h3 style='margin:0 0 10px;font-size:14px;color:#f57f17'>🗺️ Development Roadmap</h3>
        <ul style='margin:0;padding-left:18px;font-size:13px;color:#e65100;line-height:1.8'>{roadmap_li}</ul>
      </div>
    </div>

    <!-- Hire Rationale -->
    <div style='margin-top:20px;background:#fafafa;border-left:4px solid {hire_badge[1]};padding:14px 18px;border-radius:0 8px 8px 0'>
      <b style='font-size:14px'>Hire Rationale:</b>
      <p style='margin:6px 0 0;font-size:13px;color:#444;line-height:1.7'>{master_report['hire_rationale']}</p>
    </div>

  </div>

  <div style='background:#f8f9fa;padding:10px 28px;font-size:11px;color:#888;border-top:1px solid #dee2e6'>
    Generated by 3-LLM Platform Scoring Pipeline · {MODEL} · All scores out of 1000
  </div>
</div>
"""

display(HTML(html_report))

Dimension,Score,Progress,Summary
Skill Credibility,518/1000,,"The candidate shows practical AI/ML capability through end-to-end projects involving Python, Streamlit, Azure/OpenAI, YOLOv8, and measurable outcomes. Algorithmic fundamentals are decent, supported by 130 LeetCode solves and some exposure to advanced problem-solving patterns. However, the technical footprint is still closer to strong student-level experimentation than production-grade engineering, and certifications add only limited credibility."
Professional Identity + Real World Delivery,649/1000,,"The public profile is coherent, authentic, and reasonably complete, with a strong LinkedIn presence, clear AI/ML branding, and visible Microsoft Student Ambassador involvement. Real-world delivery is supported by freelance/self-employed work and concrete output claims, though much of that experience sits outside the current technical positioning. Cross-platform consistency is acceptable but not perfect, and GitHub/StackOverflow evidence remains comparatively thin."
Behavioral Consistency + Professional Track Record,756/1000,,"This is the candidate’s strongest dimension. The profile shows sustained activity, recent learning momentum, and a clear upward trajectory through Microsoft ecosystem roles and student leadership. Employment stability is generally reasonable for an early-career profile, though some short or overlapping roles create timeline complexity. Overall, the track record suggests reliability, engagement, and continued growth."


## 10 · Export JSON Report

In [33]:
final_output = {
    'metadata': {
        'model': MODEL,
        'pipeline': '3-LLM Specialist + Master Aggregator'
    },
    'llm1_skill_credibility': result_llm1,
    'llm2_identity_and_delivery': result_llm2,
    'llm3_behavioral_and_track': result_llm3,
    'aggregation': {
        'D1_skill_credibility': D1,
        'D2_identity_delivery': D2,
        'D3_behavioral_track': D3,
        'final_score': FINAL_SCORE,
        'max_score': 1000
    },
    'master_report': master_report
}

OUTPUT_FILE = f"platform_score.json"
with open(OUTPUT_FILE, 'w') as f:
    json.dump(final_output, f, indent=2, default=str)

print(f'✅  Report saved to: {OUTPUT_FILE}')
print(f'\n🏆 FINAL PLATFORM SCORE: {FINAL_SCORE} / 1000  [{master_report["tier"]}]')
print(f'📋 HIRE RECOMMENDATION:  {master_report["hire_recommendation"]}')

✅  Report saved to: platform_score.json

🏆 FINAL PLATFORM SCORE: 641 / 1000  [Strong]
📋 HIRE RECOMMENDATION:  Conditional


In [47]:
"""
╔════════════════════════════════════════════════════════════════════╔
║          HR EVALUATION PLATFORM — TRUST SCORE ENGINE            ║
║          Powered by OpenAI GPT-4o-mini w Score /1000            ║
║══════════════════════════════════════════════════════════════════════║
"""

import os, json, csv, re, time
from datetime import datetime
from pathlib import Path
from openai import OpenAI

# ── CONFIG ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_DIR   = Path(".")
OUTPUT_DIR = Path(".")
LLM_MODEL  = "gpt-5.4-mini"

# Initialize OpenAI Client
client = OpenAI(
    api_key = "CWcXBMalpGqTsxcKFF3movpCCR0xGwpQtMFEfIZEDTEd6oTsFtdlJQQJ99CDACYeBjFXJ3w3AAABACOGYHIY",
    base_url = "https://linked.openai.azure.com/openai/v1/"
)

def safe_read_csv(path_input) -> list[dict]:
    """Read a CSV file and return list-of-dicts; handle both string and Path objects."""
    path = Path(path_input)
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return [dict(row) for row in reader]

def load_candidate_bundle(data_dir: Path) -> list[dict]:
    """
    Merge all per-candidate CSVs into one rich dict per candidate.
    """
    profiles = safe_read_csv(data_dir / "candidate_profile.csv")

    if not profiles:
        # Fallback to local discovery if not in data_dir
        for fname in data_dir.iterdir():
            if "candidate_profile" in fname.name and fname.suffix == ".csv":
                profiles = safe_read_csv(fname)
                break

    if not profiles:
        raise FileNotFoundError("No candidate_profile CSV found in data_dir.")

    # Supplemental tables
    github_ctx     = safe_read_csv(data_dir / "github_contextual.csv")
    github_repos   = safe_read_csv(data_dir / "github_repos (2).csv")
    hackerrank_ctx = safe_read_csv(data_dir / "hackerrank_contextual.csv")
    leetcode_ctx   = safe_read_csv(data_dir / "leetcode_contextual.csv")
    linkedin_edu   = safe_read_csv(data_dir / "linkedin_education.csv")
    linkedin_exp   = safe_read_csv(data_dir / "linkedin_experience.csv")
    linkedin_skills= safe_read_csv(data_dir / "linkedin_skills.csv")
    resume_ctx     = safe_read_csv(data_dir / "resume_contextual.csv")
    so_ctx         = safe_read_csv(data_dir / "stackoverflow_contextual.csv")
    score_report   = safe_read_csv(data_dir / "overall_score_report.csv")

    bundles = []
    for p in profiles:
        b = dict(p)
        b["_github_context"]     = github_ctx
        b["_github_repos"]       = github_repos
        b["_hackerrank"]         = hackerrank_ctx
        b["_leetcode"]           = leetcode_ctx
        b["_linkedin_education"] = linkedin_edu
        b["_linkedin_experience"]= linkedin_exp
        b["_linkedin_skills"]    = [r.get("skill_name","") for r in linkedin_skills]
        b["_resume"]             = resume_ctx[0].get("resume_text","") if resume_ctx else ""
        b["_stackoverflow"]      = so_ctx
        b["_score_report"]       = score_report
        bundles.append(b)

    return bundles

In [48]:

# ══════════════════════════════════════════════════════════════════════════════
# 2.  BUILD LLM PROMPT
# ══════════════════════════════════════════════════════════════════════════════

def build_prompt(bundle: dict) -> str:
    """
    Convert the candidate bundle into a structured prompt that instructs the
    LLM to return a single JSON object with BOTH the candidate and recruiter
    report sections.
    """

    # ── summarise sub-tables ────────────────────────────────────────────────
    edu_lines = []
    for e in bundle.get("_linkedin_education", []):
        uni  = e.get("university_name","")
        field= e.get("fields_of_study","")
        yrs  = e.get("education_years","")
        edu_lines.append(f"  • {uni} | {field} | {yrs} yrs")

    exp_lines = []
    for e in bundle.get("_linkedin_experience", []):
        title = e.get("job_title","")
        co    = e.get("company_name","")
        months= e.get("experience_months","")
        etype = e.get("employment_type","")
        exp_lines.append(f"  • {title} @ {co}  [{etype}]  {months} months")

    repos = bundle.get("_github_repos", [])
    repo_lines = [
        f"  • {r.get('repo_name','')} | {r.get('language','')} | ⭐{r.get('stars',0)} | {r.get('description','')}"
        for r in repos[:10]
    ]

    score_lines = [
        f"  • {r.get('Platform','')} : {r.get('Platform Score','')} / 100  [{r.get('Grade','')}]"
        for r in bundle.get("_score_report", [])
    ]

    skills = ", ".join(bundle.get("_linkedin_skills", []))

    gh = bundle.get("_github_context", [{}])[0] if bundle.get("_github_context") else {}
    lc = bundle.get("_leetcode", [{}])[0]       if bundle.get("_leetcode")        else {}
    hr = bundle.get("_hackerrank", [{}])[0]     if bundle.get("_hackerrank")      else {}
    so = bundle.get("_stackoverflow", [{}])[0]  if bundle.get("_stackoverflow")   else {}

    # ── assemble context block ───────────────────────────────────────────────
    context = f"""
=== CANDIDATE PROFILE ===
Email            : {bundle.get('email','')}
Name             : {bundle.get('gh_name','') or bundle.get('lc_username','')}
Location         : {bundle.get('gh_location','') or 'N/A'}
Resume Summary   : {bundle.get('_resume','')[:800]}...

=== EDUCATION ===
{chr(10).join(edu_lines) or 'N/A'}

=== WORK EXPERIENCE ===
{chr(10).join(exp_lines) or 'N/A'}

=== LINKEDIN SKILLS ===
{skills or 'N/A'}

=== GITHUB STATS ===
Username         : {gh.get('gh_username','') or bundle.get('gh_username','')}
Bio              : {gh.get('gh_bio','')}
Followers        : {bundle.get('gh_followers',0)}
Public Repos     : {bundle.get('gh_public_repos',0)}
Languages        : {gh.get('gh_languages','') or bundle.get('gh_languages','')}
Total Stars      : {bundle.get('gh_total_stars',0)}
Account Created  : {gh.get('gh_account_created','') or bundle.get('gh_account_created','')}

=== GITHUB REPOSITORIES ({len(repos)} repos) ===
{chr(10).join(repo_lines) or 'No repos found'}

=== LEETCODE ===
Username         : {lc.get('lc_username','') or bundle.get('lc_username','')}
Problems Solved  : {bundle.get('lc_total_solved',0)} (Easy:{bundle.get('lc_easy_solved',0)} Med:{bundle.get('lc_medium_solved',0)} Hard:{bundle.get('lc_hard_solved',0)})
Contest Rating   : {bundle.get('lc_contest_rating','N/A')}
Languages        : {lc.get('lc_languages','')}
Top Topics       : {lc.get('lc_top_topics','')}

=== HACKERRANK ===
Username         : {hr.get('hr_username','') or bundle.get('hr_username','')}
Rank             : {bundle.get('hr_rank','N/A')}
Skills           : {hr.get('hr_skills_raw','')}

=== STACKOVERFLOW ===
Display Name     : {so.get('so_display_name','') or bundle.get('so_display_name','')}
Reputation       : {bundle.get('so_reputation',0)}
Answers          : {bundle.get('so_answer_count',0)}
Top Tags         : {so.get('so_top_tags','')}

=== PLATFORM SCORE REPORT ===
{chr(10).join(score_lines)}
Overall Raw Score: {bundle.get('gh_total_stars','N/A')}
"""

    # ── instructions ────────────────────────────────────────────────────────
    instructions = """
You are an expert HR evaluation AI. Analyse the candidate data above and return
ONLY a valid JSON object (no markdown fences, no extra text).

The JSON must contain exactly these two top-level keys:
  "candidate_report"  and  "recruiter_report"

────────────────────────────────────────────────────────────────
CANDIDATE REPORT keys:
────────────────────────────────────────────────────────────────
"overall_trust_score"  : integer 0-1000  (computed from multi-platform evidence)
"trust_score_breakdown": {
    "technical_skills"  : int 0-250,   // coding platforms
    "experience"        : int 0-200,   // work history quality
    "education"         : int 0-150,   // degree relevance
    "community_impact"  : int 0-150,   // SO, GitHub stars/forks
    "consistency"       : int 0-150,   // cross-platform alignment
    "fraud_penalty"     : int 0-(-100) // deduct if anomalies detected
}
"score_label"          : string  e.g. "Promising Candidate"
"what_this_score_means": string  2-3 sentence plain-English explanation
"platform_breakdown"   : list of objects:
    { "platform": str, "score": float, "max": 100, "grade": str,
      "key_finding": str, "improvement_tip": str }
"top_3_strengths"      : list of 3 strings
"areas_to_improve"     : list of 3-5 strings
"score_improvement_simulator": list of objects:
    { "action": str, "estimated_score_gain": int, "difficulty": "Easy|Medium|Hard",
      "time_estimate": str }
"salary_intelligence"  : {
    "estimated_range_usd_annual": {"min": int, "max": int},
    "market_percentile"         : int,
    "justification"             : str,
    "comparable_roles"          : list of str
}
"verified_credential_badge": {
    "badge_level"    : "Bronze|Silver|Gold|Platinum",
    "verified_items" : list of str,
    "unverified_items": list of str,
    "badge_explanation": str
}

────────────────────────────────────────────────────────────────
RECRUITER REPORT keys:
────────────────────────────────────────────────────────────────
"trust_score_summary"  : { "score": int, "label": str, "one_liner": str }
"reliability_assessment": {
    "overall_reliability" : "Low|Medium|High|Very High",
    "tenure_consistency"  : str,
    "platform_activity_trend": str,
    "red_flags"           : list of str,
    "positive_signals"    : list of str
}
"skill_verification_summary": list of objects:
    { "skill": str, "verified_by": list of str, "confidence": "Low|Medium|High" }
"fraud_risk_indicator"  : {
    "risk_level"          : "Low|Medium|High|Critical",
    "risk_score"          : int (0-100, 0=no risk),
    "anomalies_detected"  : list of str,
    "suspicious_patterns" : list of str,
    "recommendation"      : str
}
"platform_evidence_cards": list of objects per platform:
    { "platform": str, "account_age_days": int, "activity_level": str,
      "authenticity_signals": list of str, "concern_signals": list of str }
"salary_intelligence"   : {
    "recommended_offer_range_usd": {"min": int, "max": int},
    "negotiation_advice"         : str,
    "market_benchmark"           : str
}
"interview_probe_suggestions": list of objects:
    { "category": str, "question": str, "what_to_listen_for": str,
      "follow_up": str }
  (provide at least 6 probes covering: technical depth, experience gaps,
   fraud verification, behavioral, motivation, culture fit)

Be analytical, fair, and base everything strictly on the data provided.
Flag any inconsistency (e.g. short tenure listed as long, no repo activity,
very low SO reputation despite listed skills).
"""

    return context + "\n\n" + instructions


In [54]:


# ══════════════════════════════════════════════════════════════════════════════
# 3.  CALL THE LLM
# ══════════════════════════════════════════════════════════════════════════════

def call_llm(prompt: str, retries: int = 3) -> dict:
    """Send prompt to OpenAI and parse the returned JSON."""
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert HR analytics AI that outputs only valid JSON. "
                            "Never include markdown code fences or any text outside the JSON."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,

            )
            raw = response.choices[0].message.content.strip()

            # Strip accidental ``` fences
            raw = re.sub(r"^```json\s*", "", raw, flags=re.IGNORECASE)
            raw = re.sub(r"^```\s*",     "", raw, flags=re.IGNORECASE)
            raw = re.sub(r"\s*```$",     "", raw)

            return json.loads(raw)

        except json.JSONDecodeError as e:
            print(f"  ⚠  JSON parse error (attempt {attempt}): {e}")
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  ⚠  API error (attempt {attempt}): {e}")
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)


In [50]:


# ══════════════════════════════════════════════════════════════════════════════
# 4.  POST-PROCESS & ENRICH
# ══════════════════════════════════════════════════════════════════════════════

def enrich_result(result: dict, bundle: dict) -> dict:
    """Add metadata fields and normalise the score to /1000."""
    ts = result.get("candidate_report", {}).get("overall_trust_score", 0)

    result["_meta"] = {
        "candidate_email"   : bundle.get("email", "unknown"),
        "candidate_name"    : bundle.get("gh_name", "") or bundle.get("lc_username", ""),
        "evaluation_date"   : datetime.utcnow().isoformat() + "Z",
        "model_used"        : LLM_MODEL,
        "score_out_of"      : 1000,
        "final_score"       : ts,
        "score_percentage"  : round(ts / 10, 1),
    }
    return result



In [51]:

# ══════════════════════════════════════════════════════════════════════════════
# 5.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════

def save_json(data: dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved JSON → {path}")


def save_candidate_report_csv(cr: dict, meta: dict, path: Path):
    """Flatten candidate report into a single CSV row."""
    bd  = cr.get("trust_score_breakdown", {})
    sal = cr.get("salary_intelligence", {}).get("estimated_range_usd_annual", {})
    bdg = cr.get("verified_credential_badge", {})
    row = {
        "email"                  : meta.get("candidate_email"),
        "name"                   : meta.get("candidate_name"),
        "evaluation_date"        : meta.get("evaluation_date"),
        "overall_trust_score"    : cr.get("overall_trust_score"),
        "score_label"            : cr.get("score_label"),
        "score_percentage"       : meta.get("score_percentage"),
        "tech_skills_score"      : bd.get("technical_skills"),
        "experience_score"       : bd.get("experience"),
        "education_score"        : bd.get("education"),
        "community_impact_score" : bd.get("community_impact"),
        "consistency_score"      : bd.get("consistency"),
        "fraud_penalty"          : bd.get("fraud_penalty"),
        "salary_min_usd"         : sal.get("min"),
        "salary_max_usd"         : sal.get("max"),
        "market_percentile"      : cr.get("salary_intelligence", {}).get("market_percentile"),
        "badge_level"            : bdg.get("badge_level"),
        "top_strength_1"         : (cr.get("top_3_strengths") or [""])[0],
        "top_strength_2"         : (cr.get("top_3_strengths") or ["",""])[1] if len(cr.get("top_3_strengths",[])) > 1 else "",
        "top_strength_3"         : (cr.get("top_3_strengths") or ["","",""])[2] if len(cr.get("top_3_strengths",[])) > 2 else "",
        "model_used"             : meta.get("model_used"),
    }
    file_exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"  ✅ Appended to summary CSV → {path}")


def save_recruiter_report_csv(rr: dict, meta: dict, path: Path):
    """Flatten recruiter report into a single CSV row."""
    tss = rr.get("trust_score_summary", {})
    fri = rr.get("fraud_risk_indicator", {})
    sal = rr.get("salary_intelligence", {}).get("recommended_offer_range_usd", {})
    row = {
        "email"                  : meta.get("candidate_email"),
        "name"                   : meta.get("candidate_name"),
        "evaluation_date"        : meta.get("evaluation_date"),
        "recruiter_trust_score"  : tss.get("score"),
        "trust_label"            : tss.get("label"),
        "trust_one_liner"        : tss.get("one_liner"),
        "reliability_level"      : rr.get("reliability_assessment", {}).get("overall_reliability"),
        "fraud_risk_level"       : fri.get("risk_level"),
        "fraud_risk_score"       : fri.get("risk_score"),
        "fraud_recommendation"   : fri.get("recommendation"),
        "offer_min_usd"          : sal.get("min"),
        "offer_max_usd"          : sal.get("max"),
        "negotiation_advice"     : rr.get("salary_intelligence", {}).get("negotiation_advice"),
        "red_flags"              : " | ".join(rr.get("reliability_assessment", {}).get("red_flags", [])),
        "positive_signals"       : " | ".join(rr.get("reliability_assessment", {}).get("positive_signals", [])),
    }
    file_exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"  ✅ Appended to recruiter CSV → {path}")



In [52]:

# ══════════════════════════════════════════════════════════════════════════════
# 6.  PRETTY PRINT TERMINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

def print_candidate_report(cr: dict, meta: dict):
    score = cr.get("overall_trust_score", 0)
    bar_filled = int(score / 20)
    bar = "█" * bar_filled + "░" * (50 - bar_filled)

    print("\n" + "═" * 68)
    print(f"  🎯  CANDIDATE REPORT  ·  {meta.get('candidate_name','')}")
    print("═" * 68)
    print(f"\n  OVERALL TRUST SCORE:  {score} / 1000  ({meta.get('score_percentage')}%)")
    print(f"  [{bar}]")
    print(f"  Label : {cr.get('score_label','')}")
    print(f"\n  WHAT THIS MEANS\n  {cr.get('what_this_score_means','')}")

    print("\n  ── PLATFORM BREAKDOWN ──")
    for p in cr.get("platform_breakdown", []):
        print(f"  {p.get('platform',''): <14}  {p.get('score',0):>5} / {p.get('max',100)}  [{p.get('grade','')}]")
        print(f"               ↳ {p.get('key_finding','')}")

    print("\n  ── TOP 3 STRENGTHS ──")
    for i, s in enumerate(cr.get("top_3_strengths", []), 1):
        print(f"  {i}. {s}")

    print("\n  ── AREAS TO IMPROVE ──")
    for a in cr.get("areas_to_improve", []):
        print(f"  • {a}")

    print("\n  ── SCORE IMPROVEMENT SIMULATOR ──")
    for sim in cr.get("score_improvement_simulator", []):
        print(f"  +{sim.get('estimated_score_gain',0):>3} pts  [{sim.get('difficulty',''): <6}]  {sim.get('action','')}")

    sal = cr.get("salary_intelligence", {})
    rng = sal.get("estimated_range_usd_annual", {})
    print("\n  ── SALARY INTELLIGENCE ──")
    print(f"  Estimated: ${rng.get('min',0):,} – ${rng.get('max',0):,} / yr (Market {sal.get('market_percentile','')}th pct)")
    print(f"  {sal.get('justification','')}")

    bdg = cr.get("verified_credential_badge", {})
    print(f"\n  ── VERIFIED CREDENTIAL BADGE:  {bdg.get('badge_level','')} ──")
    print(f"  ✓ Verified  : {', '.join(bdg.get('verified_items',[]))}")
    print(f"  ✗ Unverified: {', '.join(bdg.get('unverified_items',[]))}")


def print_recruiter_report(rr: dict, meta: dict):
    fri = rr.get("fraud_risk_indicator", {})
    risk_icon = {"Low": "🟢", "Medium": "🟡", "High": "🔴", "Critical": "🚨"}.get(
        fri.get("risk_level", ""), "⚪"
    )

    print("\n" + "═" * 68)
    print(f"  🔍  RECRUITER REPORT  ·  {meta.get('candidate_name','')}")
    print("═" * 68)

    tss = rr.get("trust_score_summary", {})
    print(f"\n  TRUST SCORE:  {tss.get('score','')} / 1000  —  {tss.get('label','')}")
    print(f"  {tss.get('one_liner','')}")

    ra = rr.get("reliability_assessment", {})
    print(f"\n  ── RELIABILITY ASSESSMENT ──")
    print(f"  Overall : {ra.get('overall_reliability','')}")
    print(f"  Tenure  : {ra.get('tenure_consistency','')}")
    print(f"  Activity: {ra.get('platform_activity_trend','')}")
    if ra.get("red_flags"):
        print(f"  ⚠  Red Flags : {' | '.join(ra['red_flags'])}")
    if ra.get("positive_signals"):
        print(f"  ✅ Positives : {' | '.join(ra['positive_signals'])}")

    print("\n  ── SKILL VERIFICATION SUMMARY ──")
    for sv in rr.get("skill_verification_summary", []):
        print(f"  {sv.get('skill',''):<30}  [{sv.get('confidence','')}]  via {', '.join(sv.get('verified_by',[]))}")

    print(f"\n  ── FRAUD RISK INDICATOR  {risk_icon}  {fri.get('risk_level','')} ({fri.get('risk_score',0)}/100) ──")
    for a in fri.get("anomalies_detected", []):
        print(f"  ⚠  {a}")
    print(f"  → {fri.get('recommendation','')}")

    print("\n  ── INTERVIEW PROBE SUGGESTIONS ──")
    for i, q in enumerate(rr.get("interview_probe_suggestions", []), 1):
        print(f"\n  {i}. [{q.get('category','')}]")
        print(f"     Q: {q.get('question','')}")
        print(f"     Listen for: {q.get('what_to_listen_for','')}")



In [55]:
# ════════════════════════════════════════════════════════════════════════
# 7.  MAIN PIPELINE
# ════════════════════════════════════════════════════════════════════════

def evaluate_candidate(bundle: dict):
    email = bundle.get("email", "unknown").replace("@", "_").replace(".", "_")
    print(f"\n━ Evaluating: {bundle.get('email', 'unknown')}")

    prompt = build_prompt(bundle)
    print("  ┆ Calling LLM …")
    result = call_llm(prompt)
    result = enrich_result(result, bundle)

    meta = result["_meta"]
    cr   = result.get("candidate_report", {})
    rr   = result.get("recruiter_report", {})

    # ── Save full JSON reports (Ensure Path objects are used)
    save_json(result,  OUTPUT_DIR / f"full_report_{email}.json")
    save_json(cr,      OUTPUT_DIR / f"candidate_report_{email}.json")
    save_json(rr,      OUTPUT_DIR / f"recruiter_report_{email}.json")

    # ── Append to rolling CSVs
    save_candidate_report_csv(cr, meta, OUTPUT_DIR / "all_candidates_summary.csv")
    save_recruiter_report_csv(rr, meta, OUTPUT_DIR / "all_recruiter_summary.csv")

    # ── Terminal display
    print_candidate_report(cr, meta)
    print_recruiter_report(rr, meta)

    return result

def run_pipeline(data_dir: str = "."):
    """Entry point: fixed to handle Colab root directory and Path conversion."""
    # Force data_dir to current directory if Colab flag '-f' is passed
    if data_dir.startswith('-f'):
        data_dir = "."

    data_path = Path(data_dir)
    print(f"\n{'='*68}")
    print("  HR EVALUATION PLATFORM  w  Trust Score Engine v1.0")
    print(f"  Data dir : {data_path.resolve()}")
    print(f"  Output   : {OUTPUT_DIR.resolve()}")
    print(f"{'='*68}")

    bundles = load_candidate_bundle(data_path)
    print(f"  ✅ Found {len(bundles)} candidate(s).")

    results = []
    for bundle in bundles:
        try:
            r = evaluate_candidate(bundle)
            results.append(r)
        except Exception as e:
            print(f"  ❌ Error for {bundle.get('email')}: {e}")

    print(f"\n\n✅ Done. Evaluated {len(results)} / {len(bundles)} candidate(s).")
    return results

if __name__ == "__main__":
    import sys
    # Safely get directory or default to current
    user_dir = sys.argv[1] if len(sys.argv) > 1 else "."
    run_pipeline(user_dir)


  HR EVALUATION PLATFORM  w  Trust Score Engine v1.0
  Data dir : /content
  Output   : /content
  ✅ Found 1 candidate(s).

━ Evaluating: ddwivedi2003@gmail.com
  ┆ Calling LLM …
  ✅ Saved JSON → full_report_ddwivedi2003_gmail_com.json
  ✅ Saved JSON → candidate_report_ddwivedi2003_gmail_com.json
  ✅ Saved JSON → recruiter_report_ddwivedi2003_gmail_com.json
  ✅ Appended to summary CSV → all_candidates_summary.csv
  ✅ Appended to recruiter CSV → all_recruiter_summary.csv

════════════════════════════════════════════════════════════════════
  🎯  CANDIDATE REPORT  ·  Dewashish Dwivedi
════════════════════════════════════════════════════════════════════

  OVERALL TRUST SCORE:  612 / 1000  (61.2%)
  [██████████████████████████████░░░░░░░░░░░░░░░░░░░░]
  Label : Promising Candidate with Verification Gaps

  WHAT THIS MEANS
  The candidate shows real technical engagement across coding platforms, GitHub, and Microsoft/community programs, especially in ML, Python, and problem solving. However